In [10]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from langchain_chroma import Chroma
from dotenv import load_dotenv

In [3]:
def load_data(document):
    file_paths=Path(document)
    files=list(file_paths.glob("**/*pdf"))
    all_files=[]
    for i in files:
        print(f"processing {i}")
        try:
            doc=PyPDFLoader(str(i))
            data=doc.load()
            all_files.extend(data)
        except Exception as e:
            print(f"cannot open{i} due to {e}")
    return all_files

In [4]:
documents=load_data(r"E:\Datasets\land_slide_pdf")

processing E:\Datasets\land_slide_pdf\LandslideAtlas_new_2023.pdf
processing E:\Datasets\land_slide_pdf\Landslides_in_India_Issues_and_Perspective.pdf
processing E:\Datasets\land_slide_pdf\Landslide_Preparedness_Guide_.pdf


In [7]:
def text_split(document,chunk_size=1000,chunk_overlap=200):
    splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n","\n","."],
        length_function=len
    )
    doc=splitter.split_documents(document)
    return doc

In [8]:
after_split=text_split(documents)

C:\Users\user\AppData\Local\Temp\ipykernel_5480\879966086.py:1: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding=HuggingFaceBgeEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2813.01it/s]


In [ ]:
def create_db(doc):
    dir="chroma_db"
    embedding=HuggingFaceBgeEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_space=Chroma.from_documents(
        documents=doc,
        embedding=embedding,
        persist_directory=dir
    )
    return vector_space

In [12]:
vector_space=create_db(after_split)

In [ ]:
def rag_pipeline(vector_space,question):
    prompt=PromptTemplate(
        input_variables=["context","query"],
        template="this is the context:{context} answer from thi context only,if the anwer is not present then return not available in document,the question is {query},Answer:"
    )
    model=init_chat_model()